In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GIT_TOKEN_GENAI = user_secrets.get_secret("GIT_TOKEN_GENAI")

In [ ]:
import os
os.environ["GIT_TOKEN_GENAI"] = GIT_TOKEN_GENAI

In [ ]:
!git clone https://$GIT_TOKEN_GENAI@github.com/standing-on-giants/GenAI-Virtual-Try-On-System.git

In [ ]:
%cd /kaggle/working/GenAI-Virtual-Try-On-System
!pwd

In [ ]:
!sed -i 's/from diffusers.models.transformer_2d import Transformer2DModel/from diffusers.models import Transformer2DModel/g' $(grep -rl "transformer_2d" .)

!sed -i 's/from diffusers.models.dual_transformer_2d import DualTransformer2DModel/from diffusers.models import DualTransformer2DModel/g' $(grep -rl "dual_transformer_2d" .)

In [ ]:
!pip install -q \
  diffusers==0.25.0 \
  transformers==4.36.2 \
  huggingface_hub==0.20.3 \
  accelerate==0.25.0 \
  peft==0.7.1

In [ ]:
import os

# Just symlink the missing folder name (agnostic-mask -> agnostic-v3.2)
# We need a writable location for the symlink, so create a thin wrapper dir
os.makedirs("/kaggle/working/VITON-HD/test", exist_ok=True)
os.makedirs("/kaggle/working/VITON-HD/train", exist_ok=True)

DATA = "/kaggle/input/datasets/marquis03/high-resolution-viton-zalando-dataset"

# Symlink all existing folders/files into working (no copying!)
for item in os.listdir(f"{DATA}/test"):
    src = f"{DATA}/test/{item}"
    dst = f"/kaggle/working/VITON-HD/test/{item}"
    if not os.path.exists(dst):
        os.symlink(src, dst)

for item in os.listdir(f"{DATA}/train"):
    src = f"{DATA}/train/{item}"
    dst = f"/kaggle/working/VITON-HD/train/{item}"
    if not os.path.exists(dst):
        os.symlink(src, dst)

# Add the renamed mask folder as a symlink with the correct name
# os.symlink(f"{DATA}/test/agnostic-v3.2", "/kaggle/working/VITON-HD/test/agnostic-mask")

# Download and place the tagged JSONs (these are tiny)
os.system("wget -q https://raw.githubusercontent.com/yisol/IDM-VTON/main/vitonhd_test_tagged.json -O /kaggle/working/VITON-HD/test/vitonhd_test_tagged.json")
os.system("wget -q https://raw.githubusercontent.com/yisol/IDM-VTON/main/vitonhd_train_tagged.json -O /kaggle/working/VITON-HD/train/vitonhd_train_tagged.json")

# Copy the pairs txt files
import shutil
shutil.copy(f"{DATA}/test_pairs.txt", "/kaggle/working/VITON-HD/test_pairs.txt")
shutil.copy(f"{DATA}/train_pairs.txt", "/kaggle/working/VITON-HD/train_pairs.txt")

print("Done! No data copied, only symlinked.")

In [ ]:
!sed -i "s/im_name.replace('.jpg','_mask.png')/im_name.replace('.jpg','.png')/g" \
    /kaggle/working/GenAI-Virtual-Try-On-System/inference.py

!sed -i "s/im_name.replace('.jpg','.png')/im_name/g" \
    /kaggle/working/GenAI-Virtual-Try-On-System/inference.py

In [ ]:
!sed -i "s/# pipe.enable_vae_slicing()/pipe.enable_vae_slicing()/" \
    /kaggle/working/GenAI-Virtual-Try-On-System/inference.py

!sed -i "s/# pipe.enable_model_cpu_offload()/pipe.enable_model_cpu_offload()/" \
    /kaggle/working/GenAI-Virtual-Try-On-System/inference.py

!sed -i "s/image_embeds = self.unet.encoder_hid_proj(image_embeds).to(prompt_embeds.dtype)/image_embeds = self.unet.encoder_hid_proj(image_embeds.to(next(self.unet.encoder_hid_proj.parameters()).device)).to(prompt_embeds.dtype)/" \
    /kaggle/working/GenAI-Virtual-Try-On-System/src/tryon_pipeline.py

!sed -i "s/image_embeds = self.unet.encoder_hid_proj(image_embeds.to(next(self.unet.encoder_hid_proj.parameters()).device)).to(prompt_embeds.dtype)/image_embeds = self.unet.encoder_hid_proj(image_embeds.to(device=next(self.unet.encoder_hid_proj.parameters()).device, dtype=next(self.unet.encoder_hid_proj.parameters()).dtype)).to(prompt_embeds.dtype)/" \
    /kaggle/working/GenAI-Virtual-Try-On-System/src/tryon_pipeline.py

In [ ]:
#Resize all images to fit the dimensions specified via command line

filepath = "/kaggle/working/GenAI-Virtual-Try-On-System/inference.py"

with open(filepath, "r") as f:
    content = f.read()

# Fix: add .resize() to pose_img loading
old = '''        pose_img = Image.open(
            os.path.join(self.dataroot, self.phase, "image-densepose", im_name)
        )'''

new = '''        pose_img = Image.open(
            os.path.join(self.dataroot, self.phase, "image-densepose", im_name)
        ).resize((self.width, self.height))'''

content = content.replace(old, new)

with open(filepath, "w") as f:
    f.write(content)

print("Done! Verifying...")
# Verify
for i, line in enumerate(content.split('\n')):
    if 'image-densepose' in line:
        print(f"Line {i+1}: {line}")

In [ ]:
#Memory cleanup to save VRAM

filepath = "/kaggle/working/GenAI-Virtual-Try-On-System/inference.py"

with open(filepath, "r") as f:
    content = f.read()

old = '''                    for i in range(len(images)):
                        x_sample = pil_to_tensor(images[i])
                        torchvision.utils.save_image(x_sample,os.path.join(args.output_dir,sample['im_name'][i]))'''

new = '''                    for i in range(len(images)):
                        x_sample = pil_to_tensor(images[i])
                        torchvision.utils.save_image(x_sample,os.path.join(args.output_dir,sample['im_name'][i]))
                    del images, x_sample, prompt_embeds, negative_prompt_embeds
                    del pooled_prompt_embeds, negative_pooled_prompt_embeds, image_embeds
                    torch.cuda.empty_cache()
                    import gc; gc.collect()'''

content = content.replace(old, new)

In [ ]:
#To see random samples of person & cloth images together to decide which ones to keep

import os, random
from PIL import Image
import matplotlib.pyplot as plt

image_dir = "/kaggle/working/VITON-HD/test/image"
cloth_dir = "/kaggle/working/VITON-HD/test/cloth"

# Read all available pairs
with open("/kaggle/working/VITON-HD/test_pairs.txt") as f:
    all_pairs = [line.strip().split() for line in f.readlines()]

# Randomly sample N pairs to visually inspect
sample = random.sample(all_pairs, 20)

# Display them as a grid so you can pick which ones to keep
fig, axes = plt.subplots(len(sample), 2, figsize=(6, len(sample)*3))
for i, (im_name, c_name) in enumerate(sample):
    axes[i,0].imshow(Image.open(f"{image_dir}/{im_name}"))
    axes[i,0].set_title(f"Person: {im_name}", fontsize=6)
    axes[i,0].axis("off")
    axes[i,1].imshow(Image.open(f"{cloth_dir}/{c_name}"))
    axes[i,1].set_title(f"Cloth: {c_name}", fontsize=6)
    axes[i,1].axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/inspection_grid.png", dpi=150)
plt.show()

In [ ]:
# Hand-pick indices from the sample you liked
chosen_indices = [0, 1, 3, 5, 8, 9, 12, 13, 14, 16, 17, 18, 19]  # whichever you want after visual inspection
chosen_pairs = [sample[i] for i in chosen_indices]

# Write to a new pairs file
with open("/kaggle/working/VITON-HD/test_pairs.txt", "w") as f:
    for im_name, c_name in chosen_pairs:
        f.write(f"{im_name} {c_name}\n")

print(f"Will run inference on {len(chosen_pairs)} images")

In [ ]:
# # Trim pairs file to 5 images for testing
# with open("/kaggle/working/VITON-HD/test_pairs.txt") as f:
#     lines = f.readlines()
# with open("/kaggle/working/VITON-HD/test_pairs.txt", "w") as f:
#     f.writelines(lines[:5])

In [ ]:
#Replacing num_workers to 0 to make sure that dataloading works on the main process with no extra processing eating RAM
filepath = "/kaggle/working/GenAI-Virtual-Try-On-System/inference.py"

with open(filepath, "r") as f:
    content = f.read()

content = content.replace(
    "num_workers=4,",
    "num_workers=0,"
)

with open(filepath, "w") as f:
    f.write(content)

!grep -n "num_workers" /kaggle/working/GenAI-Virtual-Try-On-System/inference.py

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True accelerate launch --num_processes=1 \
    /kaggle/working/GenAI-Virtual-Try-On-System/inference.py \
    --width 768 --height 1024 --num_inference_steps 20 \
    --output_dir "/kaggle/working/result" \
    --unpaired \
    --data_dir "/kaggle/working/VITON-HD" \
    --seed 42 \
    --test_batch_size 1 \
    --guidance_scale 2.0

In [ ]:
print("hi")

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

data_dir = "/kaggle/working/VITON-HD/test"
result_dir = "/kaggle/working/result"

# Read pairs
with open("/kaggle/working/VITON-HD/test_pairs.txt") as f:
    pairs = [line.strip().split() for line in f.readlines()]

results = os.listdir(result_dir)
print(f"Results saved: {len(results)}")

# Show comparisons
fig, axes = plt.subplots(len(results), 4, figsize=(16, 5*len(results)))
if len(results) == 1:
    axes = [axes]

for i, im_name in enumerate(results):
    # Find cloth pair
    c_name = next((c for p, c in pairs if p == im_name), im_name)
    
    person = Image.open(f"{data_dir}/image/{im_name}")
    cloth  = Image.open(f"{data_dir}/cloth/{c_name}")
    mask   = Image.open(f"{data_dir}/agnostic-mask/{im_name}")
    result = Image.open(f"{result_dir}/{im_name}")
    
    for ax, img, title in zip(axes[i], 
                               [person, cloth, mask, result],
                               ["Person (Input)", "Cloth (Input)", "Mask Used", "Result"]):
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/comparison_grid.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved to /kaggle/working/comparison_grid.png")

In [ ]:
#Write code for performing inference on custom directory's images & then pushing it to GitHub

print("hi") 







In [ ]:
print("hi")